# 例 7.4 · 用主成分因子构建动量策略

如果今天推动小盘股涨跌的因子明天仍有相同的收益，哪些股票的预期收益最高？本例在标普 600 小盘股样本上，每天估计 5 个因子，按次日预期收益构建多空组合，并用下一交易日的实际收益检验预测。[MATLAB 原例](../matlab/example7_4.m) · [Python 实现](../python/examples/example7_4.py)

In [ ]:
import sys
from pathlib import Path

import numpy as np
from scipy.io import loadmat

repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / 'python' / 'epchan').is_dir()
)
sys.path.insert(0, str(repo_root / 'python'))

from epchan.backshift import backshift
from epchan.fill_missing_data import fill_missing_data
from epchan.lag1 import lag1
from epchan.ols import ols
from epchan.paths import data_path
from epchan.smartcov import smartcov
from epchan.smartmean import smartmean
from epchan.smartsum import smartsum

LOOKBACK = 252
NUM_FACTORS = 5
TOP_N = 50

## 1. 从收盘价计算日收益率

价格矩阵的每一行是一个交易日，每一列是一只股票。用上一笔可用收盘价向前填补缺失报价，再计算相邻交易日的简单收益率。股票首次报价前的缺失值会保留。

In [ ]:
data = loadmat(data_path('IJR_20080114.mat'), variable_names=['tday', 'cl'])
dates = data['tday'].ravel().astype(int)
prices = fill_missing_data(data['cl'])
previous = lag1(prices)
daily_returns = (prices - previous) / previous
print(f'{dates[0]} 至 {dates[-1]}，共 {len(dates)} 个交易日、{prices.shape[1]} 只股票。')

## 2. 滚动估计因子并选股

在交易日 $t$ 收盘后，取截至当天的 252 个日收益率，只保留窗口内收益完整的股票。对每只股票减去窗口均值，计算股票间的协方差矩阵，取最大 5 个特征值对应的特征向量作为因子暴露。

用当天去均值后的收益对因子暴露做横截面回归，得到当天的因子收益。假定它们延续一天，则每只股票的次日预期收益等于自身窗口均值加上模型拟合的因子收益贡献。预期收益最高的 50 只记为 +1，最低的 50 只记为 −1。

In [ ]:
positions = np.zeros_like(prices)
for day in range(LOOKBACK, len(dates)):
    window = daily_returns[day - LOOKBACK + 1 : day + 1]
    eligible = np.flatnonzero(np.isfinite(window).all(axis=0))
    sample = window[:, eligible]
    means = smartmean(sample, axis=0)
    centered = sample - means

    _, eigenvectors = np.linalg.eigh(smartcov(centered))
    exposures = eigenvectors[:, -NUM_FACTORS:]
    factor_returns = ols(centered[-1], exposures).beta.ravel()
    expected = means + exposures @ factor_returns

    ranked = eligible[np.argsort(expected, kind='stable')]
    positions[day, ranked[:TOP_N]] = -1
    positions[day, ranked[-TOP_N:]] = 1

## 3. 用昨日持仓计算今日损益

把持仓向后移一个交易日，再乘以当天个股收益并求和。按原例保留建仓前的空仓期，取全样本有效日损益的算术平均值，乘以 252 得到年化指标。

In [ ]:
daily_pnl = smartsum(backshift(1, positions) * daily_returns, axis=1)
annualized_mean = 252 * smartmean(daily_pnl)
print(f'年化平均损益（原例口径）= {annualized_mean:.4f}')

### 结果解读

书中报告的年化指标为 **−1.8099**。每个多头和空头各按 1 单位名义金额计，建仓时总名义敞口为 100 单位。这个负值表示，按上述头寸规模汇总后，策略在样本中的平均日损益为负。